# Completion rates in a synthetic game dataset

How do completion rates compare among games with at least the selected
number of players? All six rows are synthetic teaching values, not
measurements from Steam. Each row is one game. The completed column
counts listed players who completed it; achievements counts available
achievements and is not the completion-rate denominator.

In [ ]:
import hashlib
import platform
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

In [ ]:
datafile = "steam_achievements.csv"
minplayers = 100

In [ ]:
raw = pd.read_csv(datafile)
required = ["game", "players", "completed", "achievements"]
assert set(required) <= set(raw.columns)
assert not raw[required].isna().any().any()
counts = raw[["players", "completed", "achievements"]]
assert all(pd.api.types.is_numeric_dtype(counts[c]) for c in counts)
assert np.isfinite(counts.to_numpy()).all()
assert (counts >= 0).all().all()
assert (counts == np.floor(counts)).all().all()
assert (raw["players"] > 0).all()
assert (raw["completed"] <= raw["players"]).all()
assert raw["game"].is_unique and raw["game"].str.len().gt(0).all()
assert np.isfinite(minplayers) and minplayers >= 0
cleaned = raw.loc[raw["players"].ge(minplayers)].copy()
cleaned["rate"] = cleaned["completed"] / cleaned["players"]
cleaned = cleaned.sort_values(["rate", "game"], ascending=[False, True])

In [ ]:
display(
    Markdown(
        "## Coverage and comparison\n\n"
        f"The threshold is at least {minplayers} players per game. "
        f"This retains {len(cleaned)} of {len(raw)} games."
    )
)

In [ ]:
shown = cleaned[["game", "players", "completed", "achievements", "rate"]]
display(
    {
        "text/html": shown.to_html(
            index=False, float_format="%.3f", escape=True
        ),
        "text/latex": shown.to_latex(
            index=False, float_format="%.3f", escape=True
        ),
    },
    raw=True,
)

The completion rate divides completed by players. Counts and rates
describe different aspects of these fictional records.

In [ ]:
if cleaned.empty:
    display(
        Markdown(
            "No games meet this threshold. There are no rates to plot."
        )
    )
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(cleaned["game"], cleaned["rate"], color="#D4A017")
    ax.invert_yaxis()
    ax.set(xlim=(0, 1), xlabel="Completed / players", ylabel="Game")
    ax.set_title("Figure 1. Completion rates", fontsize=11)
    fig.tight_layout()
    plt.show()

Rates use a zero-to-one scale. No figure is drawn when no games are eligible.

In [ ]:
if not cleaned.empty:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(cleaned["game"], cleaned["players"], color="#777777")
    ax.invert_yaxis()
    ax.set(xlabel="Number of players", ylabel="Game")
    ax.set_title("Figure 2. Player counts", fontsize=11)
    fig.tight_layout()
    plt.show()

When games qualify, the player-count plot uses the same games and order
as the rate plot, making the denominators visible. Filtering changes
coverage, not the underlying per-game rate. These synthetic values support
neither population estimates nor claims about real players or games.

## Reproduction record

In [ ]:
print("Input", datafile)
print(
    "Input MD5",
    hashlib.md5(
        Path(datafile).read_bytes(), usedforsecurity=False
    ).hexdigest(),
)
print("Python", platform.python_version())
for package in [
    "pandas",
    "numpy",
    "matplotlib",
    "IPython",
    "nbconvert",
    "Jinja2",
]:
    print(package, version(package))